# RTS 実装エージェント — Qwen3.8 / Colab Pro

上から順番に実行してください。**ランタイム → ランタイムのタイプを変更 → GPU**。
初回はDrive接続・モデル約16.5GB取得・CUDAビルドに時間がかかります。Driveに20GB以上の空きを用意してください。
次回は同じDriveフォルダからモデル・Node・npmキャッシュを使います。
llama-serverはGPU/CUDA/OS/コンパイラが一致すればビルドも再利用します。
毎セッションのDrive→一時ディスクへのモデルコピーとGPUへの読込は必要です。

標準設定は **Qwen3.8-27B / Unsloth UD-Q4_K_M / 16K context / テキストのみ**。
画像入力・自動ブラウザ検査はこのNotebookの初版には含みません。
モデル取得とAPI・ツール呼び出しの確認後、「実装する」を有効にします。
GitHub SecretsはColab左側の鍵から `GH_TOKEN`（対象repoへアクセスできるtoken）を登録。
tokenはNotebook本文やDriveには保存しません。

実装Qwen → 必須チェック → 別会話の読み取り専用Qwenレビュー → 最大2回の修正・再レビューを親セルが管理します。
GPT/Codex/Claudeのレビューを前提にしません。外部投稿とPR作成は最後の専用セルで行います。
マージはユーザーが画面検査後に明示合格した場合だけ、最後の専用セルで行います。
Colabのセッション終了は防げません。作業は120秒ごと・実行終了時に保存します。
強制終了直前の最大120秒程度の変更は失われる可能性があり、途中保存は検証合格を意味しません。

In [ ]:
# @title 1. 設定（次回も同じDRIVE_FOLDERを使用）
ISSUE_NUMBER = 30 # @param {type:"integer"}
DRIVE_FOLDER = "rts-qwen-colab" # @param {type:"string"}
MODEL_REPO = "unsloth/Qwen3.8-27B-GGUF" # @param {type:"string"}
MODEL_PATTERN = "*UD-Q4_K_M*.gguf" # @param {type:"string"}
# 空欄: 初回に解決したrevisionをDriveへ固定。更新したいときだけSHAを変更。
MODEL_REVISION = "" # @param {type:"string"}
LLAMA_REF = "refs/heads/master" # @param {type:"string"}
CONTEXT_SIZE = 16384 # @param [8192,16384,32768] {type:"raw"}
REPAIR_MODEL = False # @param {type:"boolean"}
RESUME_WORK = True # @param {type:"boolean"}
# 通常は空欄。保存済みbranch/明確な既存PRを自動選択。複数候補のときだけ指定。
WORK_BRANCH = "" # @param {type:"string"}
RUN_AGENT = False # @param {type:"boolean"}
MAX_TURNS = 60 # @param {type:"integer"}
MAX_MINUTES = 45 # @param {type:"integer"}
MAX_REPAIRS = 2 # @param [0,1,2] {type:"raw"}
USER_FEEDBACK = "" # @param {type:"string"}
# MAX_MINUTESは各実装/レビュー担当の上限。全体は修正回数・テスト時間によって延びます。
REPOSITORY = "heishi1HUMANITY/rts-match-prototype"
QWEN_CODE_VERSION = "0.25.1"
assert ISSUE_NUMBER > 0 and MAX_TURNS > 0 and 1 <= MAX_MINUTES <= 180
assert DRIVE_FOLDER and "/" not in DRIVE_FOLDER and "\\" not in DRIVE_FOLDER and DRIVE_FOLDER not in (".", "..")

In [ ]:
# @title 2. DriveとGPUを確認
from google.colab import drive
from pathlib import Path
import subprocess, os
drive.mount("/content/drive")
CACHE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
RUNTIME = Path("/content/rts-agent-runtime")
CACHE.mkdir(parents=True, exist_ok=True)
RUNTIME.mkdir(parents=True, exist_ok=True)
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True)
print("GPU:", gpu.strip())
vram_mib = int(subprocess.check_output(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"], text=True).splitlines()[0])
if vram_mib < 22000:
    raise RuntimeError("標準27B設定は22GB以上のGPUが必要です。40GB以上推奨。GPUを変更して再実行してください。")
print("保存先:", CACHE)

In [ ]:
# @title 3. 必要ツールを準備（GPU/モデルには触れません）
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub>=0.34,<2"], check=True)
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "git", "gh", "ripgrep", "cmake", "build-essential", "libssl-dev"], check=True)

以下は同梱の補助処理です。編集せず実行してください。
モデルと実行環境の取得は再実行可能です。設定revisionを変えない限り同じ版を使います。

In [ ]:
"""Standard-library helpers embedded in the standalone Colab notebook."""
from __future__ import annotations

import fnmatch
from contextlib import contextmanager
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid

CHECKPOINT_LOCK = threading.Lock()


def run(args, cwd=None, env=None, capture=False):
    if capture:
        return subprocess.run(args, cwd=cwd, env=env, check=True, text=True,
                              stdout=subprocess.PIPE, stderr=subprocess.PIPE).stdout
    # Inherited OS stdout is not reliably forwarded to a Colab cell's output.
    process = subprocess.Popen(args, cwd=cwd, env=env, text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
        if code:
            raise subprocess.CalledProcessError(code, args)
    finally:
        process.stdout.close()
        if process.poll() is None:
            if os.name == "nt":
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill(); process.wait()
            else:
                stop_owned_process(process)


@contextmanager
def progress_stage(label, interval=30):
    """Display elapsed time even during a quiet download or compiler step."""
    started = time.monotonic()
    done = threading.Event()
    print(f"開始: {label}", flush=True)
    def heartbeat():
        while not done.wait(interval):
            elapsed = int(time.monotonic() - started)
            print(f"{label}: {elapsed // 60}分{elapsed % 60:02d}秒経過（処理/応答待ち。完了率ではありません）", flush=True)
    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    try:
        yield
    except BaseException:
        print(f"中断/失敗: {label}", flush=True)
        raise
    else:
        print(f"完了: {label} ({int(time.monotonic() - started)}秒)", flush=True)
    finally:
        done.set()
        worker.join(timeout=1)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp-" + uuid.uuid4().hex)
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for block in iter(lambda: source.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def safe_child(root, name):
    relative = PurePosixPath(name)
    if relative.is_absolute() or ".." in relative.parts or "\\" in name or ":" in name:
        raise ValueError(f"Unsafe relative path: {name}")
    root = Path(root).resolve()
    target = (root / name).resolve()
    if target == root or not target.is_relative_to(root):
        raise ValueError(f"Path escapes root: {name}")
    return target


def extract_files(archive, destination):
    """Restore only regular files/directories; never links or special files."""
    destination = Path(destination)
    with tarfile.open(archive, "r:gz") as source:
        members = source.getmembers()
        for member in members:
            safe_child(destination, member.name)
            if not (member.isfile() or member.isdir()):
                raise ValueError(f"Unsupported archive entry: {member.name}")
        for member in members:
            target = safe_child(destination, member.name)
            if member.isdir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with source.extractfile(member) as src, target.open("wb") as dst:
                    shutil.copyfileobj(src, dst)
                target.chmod(member.mode & 0o777)


def selected_files(siblings, pattern):
    files = sorted((item for item in siblings
                    if fnmatch.fnmatch(PurePosixPath(item["rfilename"]).name, pattern)),
                   key=lambda item: item["rfilename"])
    if not files:
        raise ValueError(f"No GGUF matches {pattern}; check the model Files page.")
    for item in files:
        safe_child(Path(tempfile.gettempdir()) / "model", item["rfilename"])
        if not item["rfilename"].endswith(".gguf") or "mmproj" in item["rfilename"].lower():
            raise ValueError("Choose language-model GGUF files only.")
        if not item.get("size") or not re.fullmatch(r"[0-9a-f]{64}", item.get("lfs", {}).get("sha256", "")):
            raise ValueError("Missing file size/SHA256 metadata.")
    names = [PurePosixPath(item["rfilename"]).name for item in files]
    parts = [re.fullmatch(r"(.+)-(\d{5})-of-(\d{5})\.gguf", name) for name in names]
    if len(files) > 1 or any(parts):
        if (not all(parts) or len({p[1] for p in parts}) != 1
                or len({p[3] for p in parts}) != 1
                or int(parts[0][3]) != len(parts)
                or sorted(int(p[2]) for p in parts) != list(range(1, len(parts) + 1))):
            raise ValueError("Ambiguous quantization or incomplete split GGUF.")
    return files


def copy_verified(source, destination, expected_hash):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and sha256(destination) == expected_hash:
        return destination
    temporary = destination.with_suffix(destination.suffix + ".partial")
    shutil.copyfile(source, temporary)
    if sha256(temporary) != expected_hash:
        temporary.unlink()
        raise ValueError("Model checksum failed; cached file may be damaged. Re-run download with REPAIR_MODEL=True.")
    temporary.replace(destination)
    return destination


def prepare_model(cache, local, repo_id, pattern, revision, repair=False):
    from huggingface_hub import HfApi, hf_hub_download
    key = hashlib.sha256(f"{repo_id}|{pattern}|{revision}".encode()).hexdigest()[:16]
    store = Path(cache) / "models" / key
    manifest = store / "manifest.json"
    if manifest.exists():
        metadata = read_json(manifest)
        print("保存済みのモデル版を再利用:", metadata["revision"])
    else:
        info = HfApi().model_info(repo_id, revision=revision or "main", files_metadata=True)
        siblings = [{"rfilename": f.rfilename, "size": f.size,
                     "lfs": {"sha256": f.lfs.sha256} if f.lfs else {}} for f in info.siblings]
        metadata = {"repo": repo_id, "revision": info.sha, "files": selected_files(siblings, pattern)}
        write_json(manifest, metadata)
    needed = sum(f["size"] for f in metadata["files"]
                 if not safe_child(store, f["rfilename"]).exists())
    local_needed = sum(f["size"] for f in metadata["files"]
                       if not safe_child(local, f["rfilename"]).exists()
                       or sha256(safe_child(local, f["rfilename"])) != f["lfs"]["sha256"])
    for directory, amount in [(store, needed), (Path(local), local_needed)]:
        directory.mkdir(parents=True, exist_ok=True)
        if shutil.disk_usage(directory).free < amount + 2 * 1024**3:
            raise RuntimeError(f"空き容量不足: {directory} / 約{amount / 1024**3:.1f} GiB + 2 GiB必要")
    result = []
    for item in metadata["files"]:
        saved = safe_child(store, item["rfilename"])
        invalid = not saved.exists() or saved.stat().st_size != item["size"]
        if repair and saved.exists():
            invalid = invalid or sha256(saved) != item["lfs"]["sha256"]
        if invalid:
            print("モデル取得:", item["rfilename"])
            downloaded = hf_hub_download(repo_id, item["rfilename"], revision=metadata["revision"],
                                         local_dir=store, force_download=repair)
            if sha256(downloaded) != item["lfs"]["sha256"]:
                raise ValueError("Downloaded model checksum mismatch")
        else:
            print("Driveの保存済みモデルを使用:", item["rfilename"])
        result.append(copy_verified(saved, safe_child(local, item["rfilename"]), item["lfs"]["sha256"]))
    return result[0], metadata


def install_node(cache, runtime):
    store = Path(cache) / "node"
    lock = store / "manifest.json"
    if lock.exists():
        metadata = read_json(lock)
    else:
        base = "https://nodejs.org/dist/latest-v24.x/"
        lines = urllib.request.urlopen(base + "SHASUMS256.txt", timeout=60).read().decode().splitlines()
        checksum, name = next(line.split() for line in lines if line.endswith("linux-x64.tar.xz"))
        version = name.removeprefix("node-").removesuffix("-linux-x64.tar.xz")
        metadata = {"name": name, "sha256": checksum, "url": f"https://nodejs.org/dist/{version}/{name}"}
        write_json(lock, metadata)
    saved = store / metadata["name"]
    if not saved.exists() or sha256(saved) != metadata["sha256"]:
        urllib.request.urlretrieve(metadata["url"], saved.with_suffix(".partial"))
        saved.with_suffix(".partial").replace(saved)
    if sha256(saved) != metadata["sha256"]:
        raise ValueError("Node archive checksum mismatch")
    destination = Path(runtime) / "node"
    destination.mkdir(parents=True, exist_ok=True)
    run(["tar", "-xJf", str(saved), "--strip-components=1", "-C", str(destination)])
    os.environ["PATH"] = str(destination / "bin") + os.pathsep + os.environ["PATH"]
    os.environ["npm_config_cache"] = str(Path(cache) / "npm-cache")
    run(["node", "--version"])


def build_server(cache, runtime, requested_ref):
    nvcc = run(["nvcc", "--version"], capture=True)
    capability = run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], capture=True).splitlines()[0].strip().replace(".", "")
    compiler = run(["c++", "--version"], capture=True).splitlines()[0]
    system = Path("/etc/os-release").read_text()
    lock = Path(cache) / "llama" / "source.json"
    if lock.exists() and read_json(lock)["requested_ref"] == requested_ref:
        commit = read_json(lock)["commit"]
    else:
        commit = run(["git", "ls-remote", "https://github.com/ggml-org/llama.cpp.git", requested_ref], capture=True).split()[0]
        write_json(lock, {"requested_ref": requested_ref, "commit": commit})
    fingerprint = hashlib.sha256(f"{commit}|{nvcc}|{capability}|{compiler}|{system}".encode()).hexdigest()[:20]
    archive = Path(cache) / "llama" / f"{fingerprint}.tar.gz"
    output = Path(runtime) / "llama" / fingerprint
    binary = output / "bin" / "llama-server"
    if archive.exists():
        extract_files(archive, output)
        try:
            run([str(binary), "--version"], capture=True)
            print("GPU/CUDA環境が一致: llama-serverのビルドを再利用")
            return binary, commit
        except (OSError, subprocess.CalledProcessError):
            print("保存済み実行ファイルを使えないため再ビルドします")
    source = Path(runtime) / "llama-source"
    if not (source / ".git").exists():
        run(["git", "clone", "--filter=blob:none", "https://github.com/ggml-org/llama.cpp.git", str(source)])
    run(["git", "fetch", "origin", commit], cwd=source)
    run(["git", "checkout", "--detach", commit], cwd=source)
    run(["cmake", "-S", str(source), "-B", str(output), "-DGGML_CUDA=ON",
         f"-DCMAKE_CUDA_ARCHITECTURES={capability}", "-DGGML_NATIVE=OFF",
         "-DBUILD_SHARED_LIBS=OFF", "-DLLAMA_CURL=OFF", "-DCMAKE_BUILD_TYPE=Release"])
    run(["cmake", "--build", str(output), "--config", "Release", "-j", "2", "--target", "llama-server"])
    archive.parent.mkdir(parents=True, exist_ok=True)
    temporary = archive.with_suffix(".partial")
    with tarfile.open(temporary, "w:gz") as tar:
        tar.add(output / "bin", arcname="bin")
    temporary.replace(archive)
    return binary, commit


def checkpoint(repo, destination, details=None):
    with CHECKPOINT_LOCK:
        return _checkpoint(repo, destination, details)


def _checkpoint(repo, destination, details=None):
    repo, destination = Path(repo), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    checkpoint_id = time.strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:8]
    folder = destination / checkpoint_id
    folder.mkdir()
    head = run(["git", "rev-parse", "HEAD"], cwd=repo, capture=True).strip()
    branch = run(["git", "branch", "--show-current"], cwd=repo, capture=True).strip()
    if not branch:
        raise RuntimeError("Checkpoint requires a named branch")
    run(["git", "bundle", "create", str(folder / "repo.bundle"), "--all"], cwd=repo, capture=True)
    patch = subprocess.check_output(["git", "diff", "--binary", "HEAD"], cwd=repo)
    (folder / "changes.patch").write_bytes(patch)
    names = subprocess.check_output(["git", "ls-files", "--others", "--exclude-standard", "-z"], cwd=repo).decode().split("\0")
    with tarfile.open(folder / "untracked.tar.gz", "w:gz") as tar:
        for name in filter(None, names):
            path = safe_child(repo, name)
            if path.is_symlink() or not path.is_file():
                raise ValueError(f"Unsupported untracked file: {name}")
            tar.add(path, arcname=name, recursive=False)
    if run(["git", "rev-parse", "HEAD"], cwd=repo, capture=True).strip() != head:
        raise RuntimeError("HEAD changed during checkpoint; retry")
    metadata = {"id": checkpoint_id, "head": head, "branch": branch, "details": details or {}}
    write_json(folder / "manifest.json", metadata)
    write_json(destination / "latest.json", metadata)
    return folder


def restore_checkpoint(destination, repo, remote):
    destination, repo = Path(destination), Path(repo)
    if repo.exists():
        raise RuntimeError("Existing workspace is preserved; restore into an empty path")
    metadata = read_json(destination / "latest.json")
    folder = safe_child(destination, metadata["id"])
    run(["git", "clone", "--branch", metadata["branch"], str(folder / "repo.bundle"), str(repo)], capture=True)
    run(["git", "remote", "set-url", "origin", remote], cwd=repo)
    if run(["git", "rev-parse", "HEAD"], cwd=repo, capture=True).strip() != metadata["head"]:
        raise ValueError("Checkpoint HEAD mismatch")
    patch = folder / "changes.patch"
    if patch.stat().st_size:
        run(["git", "apply", "--binary", str(patch)], cwd=repo)
    extract_files(folder / "untracked.tar.gz", repo)
    return metadata


def agent_settings(context_size, reviewer=False):
    settings = {
        "security": {"auth": {"selectedType": "openai"}},
        "model": {"name": "qwen-local"},
        "modelProviders": {"openai": [{"id": "qwen-local", "baseUrl": "http://127.0.0.1:8080/v1",
            "envKey": "LOCAL_LLM_API_KEY", "generationConfig": {"contextWindowSize": context_size,
            "timeout": 600000, "samplingParams": {"temperature": 1.0, "top_p": 0.95, "max_tokens": 4096}}}]},
        "context": {"fileName": ["AGENTS.md", "QWEN.md"]},
        "tools": {"approvalMode": "auto-edit", "core": ["read_file", "read_many_files",
            "list_directory", "grep_search", "glob", "write_file", "edit", "run_shell_command", "todo_write"]},
        "permissions": {"allow": ["run_shell_command(npm run check)", "run_shell_command(npm run references:check)",
            "run_shell_command(npm run format)", "run_shell_command(npm run typecheck)", "run_shell_command(npm test *)",
            "run_shell_command(npm run build)", "run_shell_command(git status *)", "run_shell_command(git diff *)", "run_shell_command(git log *)", "run_shell_command(rg *)"],
            "deny": ["run_shell_command(git push *)", "run_shell_command(gh *)", "run_shell_command(rm -rf *)"]},
        "memory": {"enableManagedAutoMemory": False, "enableManagedAutoDream": False, "enableAutoSkill": False},
        "telemetry": {"enabled": False}}
    if reviewer:
        settings["tools"] = {"approvalMode": "plan", "core": [
            "read_file", "read_many_files", "list_directory", "grep_search", "glob"]}
        settings["permissions"] = {"allow": [], "deny": [
            "run_shell_command", "write_file", "edit", "agent", "web_fetch", "web_search"]}
    return settings


def agent_command(prompt, turns=60, minutes=45, reviewer=False):
    command = ["qwen", "--auth-type", "openai", "--model", "qwen-local", "--approval-mode", "plan" if reviewer else "auto-edit",
            "--max-session-turns", str(turns), "--max-tool-calls", "150", "--max-wall-time", f"{minutes}m",
            "--output-format", "stream-json", "--include-partial-messages"]
    excluded = ["agent", "workflow", "exec", "skill", "tool_call", "tool_search",
        "web_fetch", "web_search", "enter_worktree", "exit_worktree", "record_artifact",
        "report_findings", "send_message", "task_stop", "update_goal", "get_goal", "list_agents"]
    if reviewer:
        command += ["--json-schema", json.dumps(REVIEW_SCHEMA)]
        excluded += ["write_file", "edit", "notebook_edit", "run_shell_command", "todo_write"]
    command += ["--exclude-tools", ",".join(excluded)]
    command += ["--prompt", prompt]
    return command


REVIEW_SCHEMA = {
    "type": "object", "additionalProperties": False,
    "properties": {
        "status": {"type": "string", "enum": ["pass", "changes_required", "blocked"]},
        "findings": {"type": "array", "items": {
            "type": "object", "additionalProperties": False,
            "properties": {"id": {"type": "string"}, "file": {"type": "string"},
                "severity": {"type": "string", "enum": ["high", "medium", "low"]},
                "problem": {"type": "string"}, "expected": {"type": "string"},
                "verification": {"type": "string"}},
            "required": ["id", "file", "severity", "problem", "expected", "verification"]}},
        "unchecked": {"type": "array", "items": {"type": "string"}},
        "user_checks": {"type": "array", "minItems": 3, "maxItems": 5,
            "items": {"type": "object", "additionalProperties": False,
                "properties": {"action": {"type": "string"}, "expected": {"type": "string"}},
                "required": ["action", "expected"]}}},
    "required": ["status", "findings", "unchecked", "user_checks"]}


def review_ready(report):
    """Fail closed even when a model contradicts its own pass verdict."""
    if not isinstance(report, dict) or set(report) != set(REVIEW_SCHEMA["required"]):
        return False
    if report["status"] != "pass" or report["findings"] != []:
        return False
    if not isinstance(report["unchecked"], list) or not all(isinstance(s, str) for s in report["unchecked"]):
        return False
    checks = report["user_checks"]
    return (isinstance(checks, list) and 3 <= len(checks) <= 5
            and all(isinstance(c, dict) and set(c) == {"action", "expected"}
                    and all(isinstance(v, str) and v.strip() for v in c.values()) for c in checks))


def workspace_fingerprint(repo):
    """Bind verification to file contents, including deletions and untracked files.

    Commits with unchanged contents keep the same fingerprint. Ignored build/cache
    files do not invalidate the source review.
    """
    names = subprocess.check_output(["git", "ls-files", "--cached", "--others", "--exclude-standard", "-z"], cwd=repo).decode().split("\0")
    digest = hashlib.sha256()
    for name in sorted(set(filter(None, names))):
        if (Path(repo) / name).is_symlink():
            raise ValueError(f"Review does not support symbolic links: {name}")
        path = safe_child(repo, name)
        if not path.is_file():
            continue
        digest.update(name.encode() + b"\0")
        digest.update(sha256(path).encode() + b"\0")
    return digest.hexdigest()


def prepare_control_folder(repo):
    folder = Path(repo) / ".colab"
    folder.mkdir(exist_ok=True)
    git_dir = Path(run(["git", "rev-parse", "--absolute-git-dir"], cwd=repo, capture=True).strip())
    exclude = git_dir / "info/exclude"
    exclude.parent.mkdir(exist_ok=True)
    previous = exclude.read_text(encoding="utf-8") if exclude.exists() else ""
    if "/.colab/" not in previous.splitlines():
        exclude.write_text(previous + "\n/.colab/\n", encoding="utf-8")
    return folder


def choose_work_branch(issue_number, requested, saved, related):
    if requested:
        return requested
    candidates = {p["headRefName"] for p in related}
    default = f"colab/issue-{issue_number}"
    if saved and (not candidates or saved in candidates):
        return saved
    if default in candidates or not candidates:
        return default
    if len(candidates) == 1:
        return next(iter(candidates))
    raise RuntimeError("関連PRが複数あり継続先を選べません。WORK_BRANCHで対象を指定してください。")


def run_qwen_phase(repo, destination, prompt, env, context_size, turns, minutes, reviewer=False):
    """One fresh CLI conversation; never resumes the implementation's transcript."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    settings_path = Path.home() / ".qwen/settings.json"
    write_json(settings_path, agent_settings(context_size, reviewer))
    process = None
    result = None
    denials = []
    try:
        process = subprocess.Popen(agent_command(prompt, turns, minutes, reviewer), cwd=repo,
            env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
            start_new_session=True)
        with (destination / "events.jsonl").open("w", encoding="utf-8") as log:
            for line in process.stdout:
                clean = line.replace(env.get("LOCAL_LLM_API_KEY", "__missing_key__"), "[redacted]")
                log.write(clean); log.flush()
                print(clean, end="")
                try:
                    event = json.loads(clean)
                    if event.get("type") == "result":
                        result = event
                        denials.extend(event.get("permission_denials") or [])
                except (ValueError, AttributeError):
                    pass
        exit_code = process.wait()
    finally:
        if process is not None and process.poll() is None:
            stop_owned_process(process)
        write_json(settings_path, agent_settings(context_size))
    success = (exit_code == 0 and isinstance(result, dict) and not result.get("is_error")
               and result.get("subtype") == "success" and not denials)
    report = None
    if reviewer and result:
        report = result.get("structured_result")
        if report is None:
            try:
                report = json.loads(result.get("result", ""))
            except (ValueError, TypeError):
                pass
    summary = {"success": success, "exit_code": exit_code, "permission_denials": denials, "report": report}
    write_json(destination / "result.json", summary)
    return summary


def run_required_checks(repo, destination):
    results = []
    for command in (["npm", "run", "check"], ["npm", "run", "references:check"]):
        process = subprocess.run(command, cwd=repo, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True)
        results.append({"command": command, "exit_code": process.returncode, "output": process.stdout[-12000:]})
        print(process.stdout[-12000:])
    write_json(Path(destination) / "checks.json", results)
    return results


def checks_passed(checks):
    required = {("npm", "run", "check"), ("npm", "run", "references:check")}
    return (isinstance(checks, list) and len(checks) == len(required)
            and all(isinstance(c, dict) and c.get("exit_code") == 0
                    and isinstance(c.get("command"), list) for c in checks)
            and {tuple(c.get("command", [])) for c in checks} == required)


def run_issue_workflow(repo, destination, issue_number, issue_context, env, context_size,
                       turns=60, minutes=45, repairs=2, feedback="",
                       phase_runner=run_qwen_phase, checker=run_required_checks):
    """Python owns the required implement -> review -> fix -> re-review loop."""
    repo, destination = Path(repo), Path(destination)
    if repairs not in (0, 1, 2):
        raise ValueError("At most two automatic repair rounds are supported")
    destination.mkdir(parents=True, exist_ok=True)
    control = prepare_control_folder(repo)
    state_path = destination / "workflow.json"
    # Immediately revoke an earlier successful run before any new edits/checks.
    state = {"ready": False, "issue": issue_number,
        "branch": run(["git", "branch", "--show-current"], cwd=repo, capture=True).strip(),
        "phase": "implementing", "rounds": []}
    write_json(state_path, state)
    common = (f"Issue #{issue_number}。作業場所 {repo}。AGENTS.mdと.colab/workflow.md、"
        "docs/project/ISSUE_AGENT_RUNBOOK.mdとISSUE_IMPLEMENTATION_GUIDE.mdの対象節を読む。"
        f"最新Issue/コメント/関連PRは {issue_context}。.colab/ci-failures.jsonがあればCI失敗ログも読み修正する。既存差分は保全して継続する。"
        "利用者へ技術設計やコードレビューを要求しない。GitHub投稿/push/merge/Issue closeは親セル専用。"
        "実施していないテスト/ブラウザ操作/ユーザー合格を実施済みと書かない。")
    request = common + ("設計・実装・テスト・ユーザー用3〜5操作と期待結果を.COLAB_HANDOFF.mdへ記録。"
        "範囲を内部APIだけに縮小せず、通常対戦へ接続する。" + ("\nユーザーの追加指示:\n" + feedback if feedback.strip() else ""))
    for attempt in range(repairs + 1):
        round_dir = destination / f"round-{attempt + 1}"
        implementation = phase_runner(repo, round_dir / "implementation", request, env, context_size, turns, minutes)
        state["phase"] = "checking"
        write_json(state_path, state)
        checks = checker(repo, round_dir)
        fingerprint = workspace_fingerprint(repo)
        input_path = control / "review-input.json"
        base = run(["git", "merge-base", "origin/main", "HEAD"], cwd=repo, capture=True).strip()
        diff = subprocess.check_output(["git", "diff", "--binary", base], cwd=repo).decode(errors="replace")
        (control / "review.diff").write_text(diff, encoding="utf-8")
        write_json(input_path, {"issue_context": str(issue_context), "base": base,
            "diff_file": str(control / "review.diff"), "files": run(["git", "status", "--short"], cwd=repo, capture=True),
            "implementation": implementation, "checks": checks, "fingerprint": fingerprint,
            "feedback": feedback, "previous_rounds": state["rounds"]})
        state["phase"] = "reviewing"
        write_json(state_path, state)
        review_prompt = common + (f"\nあなたは実装者と別会話の読み取り専用レビュー担当。{input_path}とreview.diff、"
            "変更/追加ファイル、実際のテストを読む。通常対戦への接続、仕様・受入条件、境界、NPC公平性、"
            "費用/視界/command、マスタ契約、回帰、テストの検出力を独立に検証。"
            "実装者の自己申告を合格の根拠にしない。コード編集やshell実行はしない。"
            "現物で確認した問題はseverity/file/problem/expected/verification付きfindingsへ。"
            "未解決問題やテスト失敗はchanges_required、必要資料不足はblocked。"
            "findingsゼロかつ全必須チェック成功のときだけpass。見た目の実機未検査はuncheckedへ明示。"
            "user_checksは利用者が画面で試せる3〜5操作と期待結果。指定schemaのJSONを出力。")
        review = phase_runner(repo, round_dir / "review", review_prompt, env, context_size,
            turns, minutes, reviewer=True)
        unchanged = workspace_fingerprint(repo) == fingerprint
        passed = (implementation["success"] and review["success"] and unchanged
            and checks_passed(checks) and review_ready(review.get("report"))
            and (repo / ".COLAB_HANDOFF.md").is_file())
        record = {"implementation_success": implementation["success"], "review": review,
            "checks_passed": checks_passed(checks), "source_unchanged": unchanged}
        state["rounds"].append(record)
        state.update({"ready": passed, "fingerprint": fingerprint, "phase": "ready" if passed else "needs-fix"})
        write_json(state_path, state)
        if passed:
            return state
        if not unchanged:
            state["phase"] = "review-modified-workspace"
            write_json(state_path, state)
            return state
        request = common + (f"\n前回のチェック/独立レビュー結果は {input_path} と "
            f"{round_dir / 'review/result.json'}。指摘と失敗を修正し、.COLAB_HANDOFF.mdを更新する。"
            "レビュー結果を編集して合格にしない。必要な根拠でコード/テストを直す。")
    return state


def require_reviewed_workspace(repo, state, issue_number=None):
    if (not state.get("ready") or workspace_fingerprint(repo) != state.get("fingerprint")
            or (issue_number is not None and state.get("issue") != issue_number)
            or (state.get("branch") is not None and state["branch"] != run(
                ["git", "branch", "--show-current"], cwd=repo, capture=True).strip())):
        raise RuntimeError("独立レビュー未完了、またはレビュー後に内容が変わりました。実装セルで再検証してください。")


def require_approved_revision(repo, state, issue_number, preview, published_sha, pr):
    require_reviewed_workspace(repo, state, issue_number)
    expected = run(["git", "rev-parse", "HEAD"], cwd=repo, capture=True).strip()
    if preview.get("dirty") is not False or preview.get("commit") != expected or published_sha != expected:
        raise RuntimeError("PR提出後のclean版をセル12で画面確認してください。")
    if pr.get("headRefOid") != expected:
        raise RuntimeError("PRの内容が検査版から変わっています。再検査してください。")
    checks = pr.get("statusCheckRollup") or []
    required_names = {"checks (ubuntu-latest)", "checks (windows-latest)"}
    successful = {c.get("name") for c in checks if c.get("conclusion") == "SUCCESS"}
    if not required_names <= successful or not all(
            c.get("conclusion") in ("SUCCESS", "SKIPPED", "NEUTRAL") or c.get("state") == "SUCCESS" for c in checks):
        raise RuntimeError("Windows/Linux CI成功待ち、または未完了/失敗のチェックがあります。")
    return expected


def stop_owned_process(process):
    """Only for processes launched by this notebook with start_new_session=True."""
    if process is None or process.poll() is not None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
        process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)
        process.wait()
    except ProcessLookupError:
        process.wait()

In [ ]:
# Qwenだけで実装・レビュー・検査準備を進める共通指示
QWEN_WORKFLOW_INSTRUCTIONS = '# Issue番号からQwenだけで進める作業指示\n\nこの文書は、GPT6luna/Codexを実装者・技術レビュー担当に要求せず、ColabのQwenでIssueを検査可能な版まで進めるための恒久指示。利用者はIssue番号を指定し、提出された画面のふるまいを確認する。設計書・コード・テスト結果の技術レビューを利用者へ移さない。\n\n仕様の優先順位とゲームの完了条件は[AGENTS](../../AGENTS.md)、[Issue作業手順](ISSUE_AGENT_RUNBOOK.md)、[WORKFLOW](WORKFLOW.md)に従う。この文書はColabでの担当と実行順を具体化する。以前の会話履歴、特定PCのパス、Codex専用ツール、外部モデルへの委任を前提にしない。\n\n## 担当と順番\n\nNotebookが親として最新Issueの取得、作業保存、役割の起動、チェック、GitHubへの提出を管理する。実装とレビューは同じGPU・同じQwenモデルを順番に使用する。レビューは別プロセス・別会話で起動し、実装者の自己採点を引き継がない。ただし、同じモデルによるレビューなので別モデルの独立性や正しさの保証とは異なる。\n\n1. 親がIssueの最新本文・コメント・関連PRとブランチを照合する。\n2. 実装Qwenが設計、実装、テスト、検査手順を用意する。\n3. 親が検証結果と対象差分を記録する。\n4. 読み取り専用のレビューQwenが仕様と差分を独立に確認する。\n5. 指摘があれば実装Qwenへ戻し、修正・再検証・新しい会話での再レビューを行う。自動修正回数は`MAX_REPAIRS`で指定し、標準は最大2回。`MAX_MINUTES`は各担当の上限で、全体には修正とテストの時間も加わる。残る問題を隠して提出可能としない。\n6. 問題が解消した対象差分だけを、親の提出セルから専用ブランチのdraft PRへ提出する。\n7. 利用者へ起動方法と短い検査手順を渡す。ゲーム機能のマージは対象版への明示合格後。\n\nQwen Codeのネイティブなサブエージェント定義は補助として使用できるが、実装者が自主的に呼ぶことだけを必須レビューの保証にしない。Notebookの別プロセスによるレビューと提出判定を必ず通す。\n\n## 実装Qwenへの指示\n\nあなたは対象Issueを、通常試合で使えて利用者が画面検査できる区切りまで実装する。調査報告、内部API、固定fixtureだけで終了しない。以下を順番に実行する。\n\n1. remote、現在のブランチ、HEAD、既存差分を確認する。対象は`heishi1HUMANITY/rts-match-prototype`。既存変更を破棄・巻き戻ししない。閉じたIssueやマージ済み機能を再実装せず、既存PRの修正なら同じブランチを継続する。\n2. AGENTS、README、SCOPE、ARCHITECTURE、DECISIONS、WORKFLOW、TASKSを読む。[実装ガイド](ISSUE_IMPLEMENTATION_GUIDE.md)は冒頭の「着手と完了の扱い」「実装基準と共通契約」「依存の読み方」と対象Issue節を先に読み、リンク先の設計・網羅表・必要な旧資料を開く。全Issueを一度にコンテキストへ詰め込まない。\n3. 親が渡した最新Issue本文・コメント・関連PRを読む。最新ユーザー指示を優先し、過去の引き継ぎから現在のマージ状況・テスト件数を推測しない。取得できていない情報を「確認済み」と書かない。\n4. 利用者に見える変化、受入条件、必須依存、変更する状態・command・UI・AI・診断を整理する。通常の技術選択は自分で決め、数値の単位・理由・暫定性を設計とDECISIONSへ残す。採用機能や承認済みルールを実質的に変える判断だけを、短い遊びの比較として利用者へ返す。\n5. ルール・入力・表示・NPC・視界・診断を接続する。simulationはJSON化可能な固定tick/seedとし、DOMや壁時計に依存させない。人間とB/Cは同じcommand検証、費用、研究、通行、視界を使う。中立NPCの特例は別に明示し、B/Cへ無料生産や不可視情報を与えない。\n6. 調整値は現在の試合に適用中の不変マスタへ接続する。編集・保存値と実行中マスタを混同しない。F17未統合なら現行契約と未接続を明記し、先行機能を無断で混ぜない。研究充填・未指定バンク・進捗保持・基本所得、ドラッグ・複数選択・部分受理を維持する。\n7. 正常系だけでなく、拒否時state不変、費用/報酬の一回性、境界値、同tick競合、視界外情報、pause/終了/reset、JSON往復とreplayを変更に応じて検証する。失敗したテストを削除・弱体化して成功扱いしない。\n8. `npm run check`と`npm run references:check`を実行する。実行禁止・失敗・未実施を成功と混ぜない。ブラウザ操作が使えなければ検査手順とプレビュー準備を行い、「画面確認待ち」と記録する。HTTP応答だけでは画面確認にならない。\n9. `.COLAB_HANDOFF.md`を更新する。対象Issue、基点、変更、完了条件ごとの結果、実施済み/失敗/未実施チェック、既知の問題、未完了、ユーザーが開く画面・操作・期待結果、再開時の次の一手を残す。作業を止める前にも保存する。\n\n実装者はGitHub tokenを使わない。push、PR/Issue投稿、ラベル変更、マージ、Issue close、外部公開、権限回避を行わない。必要な外部記録を引き継ぎへ書き、親の認証済みセルに任せる。許可されないコマンドは別経路で迂回せず、必要性と未実施内容を報告する。\n\n## 読み取り専用レビューQwenへの指示\n\nあなたは実装者とは別会話の技術レビュー担当。対象Issue、共通契約、実コード、基点からの差分、親が記録したチェック結果を根拠にする。実装者の「完了」「問題なし」を証拠として採用しない。\n\n- ファイルの編集、commit、依存導入、テスト結果の作成、GitHub投稿をしない。読み取り可能な証拠から確認し、実行できない検査は未確認として返す。\n- 受入条件への接続漏れ、通常試合で使えない機能、NPCの不正な情報/費用、費用の重複、視界漏洩、マスタ参照漏れ、tick順、reset/replay、操作取消・無効理由を重点確認する。\n- 変更の影響を受ける既存機能と意味のあるテストを確認する。好みの命名や無関係な全面改修をblockerにしない。\n- 指摘は対象ファイル/箇所、具体的な操作や入力、期待結果と実際の結果、影響、優先度、解消条件を示す。再現やコード根拠がない推測を確定バグにしない。\n- 親が指定するJSON形式で結果を返す。対象差分、blocking指摘、残る懸念、未確認範囲を含める。構文不正、欠落、対象差分の不一致はレビュー合格ではない。\n\nレビューはコードの読解であり、実ブラウザ確認・実モデルの品質評価・ユーザー合格の代わりにはならない。blocking指摘が残るなら、不合格のまま具体的な残作業を返す。\n\n## 検査準備と提出の指示\n\n親は実装ログ、レビューJSON、対象差分のfingerprint、チェック結果を照合する。修正・復元・ブランチ切替後に以前の合格を流用しない。終了コード0やレビュー結果の文章だけでは提出を許可しない。証拠が欠ける場合やblocking指摘が残る場合は作業を保存し、未完了として報告する。部分成果をdraftに残す場合も「検査可能」「技術確認済み」に偽装しない。\n\n通常の提出では、専用ブランチへcommitし、cleanな対象SHAからbuildする。`dist/build-info.json`のSHA/dirty/schema/rulesを照合し、CI結果は実際のURLと状態で記録する。ローカル成功をWindows/Linux CI成功と書かない。PRは関連Issueを示すが、ユーザー合格前に自動Closeを付けない。\n\n[検査依頼テンプレート](templates/HANDOFF.md)を埋め、利用者には次だけを短く示す。\n\n- 実際に開く画面と、開けない場合の起動方法。\n- 今回できること、通常設定と短時間fixtureの違い。\n- 3〜5個の操作と、それぞれ画面で分かる期待結果。\n- 既知の問題と未検査範囲。PC/900×420/タッチの実機・エミュレーションを区別する。\n- 検査版IDと対象SHA。検査中はその版を変更しない。\n\nColabのプレビューはそのNotebookセッション内の画面で、別PCのlocalhostから見えると推測しない。初版のテキストモデル構成には自動ブラウザ操作を含めない。GPU起動や実プレイを未実施の環境から確認済みと宣言しない。\n\n## 修正、停止再開、合格\n\n利用者の修正指示は「#15、橋を押しても進まない。隣の岸に渡ってほしい」のような普通の言葉でよい。セル1の`USER_FEEDBACK`へ入力しセル10から再実行する。親がその原文と対象検査版を実装Qwenへ渡し、エージェントが再現条件・期待結果・修正方針を整理する。利用者に技術的な原因やコードを要求しない。修正後は同じIssue/PRで検証と独立レビューをやり直し、新しい検査版を渡す。\n\n再開時は同じDriveフォルダ、Issue、ブランチを使用する。保存したコードと`.COLAB_HANDOFF.md`、最新Issueから再開し、会話記憶を前提にしない。最新mainと既存PRを再照合し、復元した古いレビューやチェック結果はそのまま提出根拠にしない。強制終了で失われた変更を推測で復元済みと報告しない。\n\nPR提出後にセル12を再実行し、commit済みのcleanな対象版で画面を確認する。合格した場合だけ任意のセル14で`USER_ACCEPTED=True`を指定する。認証済みの親セルがプレビューSHA、提出SHA、現在のPR headの一致、Windows/Linux CI成功、競合を照合してマージし、その後Issueを閉じる。版の不一致やCI待ちでは停止する。実装・提出セルは自動マージしない。Codexのレビューや操作は必須にしないが、ユーザー合格とマージ権限は省略しない。\n\nチャットなどで「検査OK、マージして」を受けた場合も最後に提示した版への合格として記録し、同じSHA照合を行う。複数版が候補にあるときだけ対象を確認する。合格後にコードや競合解消で内容が変わった場合は、検証・レビュー・ユーザー検査へ戻す。\n'

In [ ]:
# @title 4. Node / Qwen Code / CUDA実行プログラム
with progress_stage("Nodeの取得・検証・展開"):
    install_node(CACHE, RUNTIME)
with progress_stage("Qwen Codeのインストール"):
    run(["npm", "install", "--global", f"@qwen-code/qwen-code@{QWEN_CODE_VERSION}"])
with progress_stage("CUDA実行プログラムの復元/初回ビルド"):
    SERVER_BINARY, LLAMA_COMMIT = build_server(CACHE, RUNTIME, LLAMA_REF)
print("llama.cpp revision:", LLAMA_COMMIT)
run(["qwen", "--version"])

In [ ]:
# @title 5. モデルをDriveに取得し一時ディスクへコピー
MODEL_PATH, MODEL_METADATA = prepare_model(CACHE, RUNTIME / "models", MODEL_REPO,
                                         MODEL_PATTERN, MODEL_REVISION, REPAIR_MODEL)
print("モデル:", MODEL_PATH.name, "revision:", MODEL_METADATA["revision"])

In [ ]:
# @title 6. ローカルAPIを起動（外部へ公開しません）
import secrets, socket, urllib.error
if "server_process" in globals() and server_process.poll() is None:
    stop_owned_process(server_process)
with socket.socket() as test_socket:
    test_socket.bind(("127.0.0.1", 8080))
LOCAL_LLM_API_KEY = secrets.token_urlsafe(32)
server_log = open(RUNTIME / "llama-server.log", "w", encoding="utf-8")
server_process = subprocess.Popen([
    str(SERVER_BINARY), "--model", str(MODEL_PATH), "--alias", "qwen-local",
    "--host", "127.0.0.1", "--port", "8080", "--api-key", LOCAL_LLM_API_KEY,
    "--n-gpu-layers", "99", "--ctx-size", str(CONTEXT_SIZE), "--parallel", "1",
    "--jinja", "--reasoning-format", "deepseek", "--reasoning-budget", "2048",
    "--chat-template-kwargs", json.dumps({"enable_thinking": True, "reasoning_effort": "medium"}),
], stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True)
deadline = time.monotonic() + 600
while True:
    if server_process.poll() is not None:
        raise RuntimeError("モデル起動失敗。次の診断セルでログを確認してください。")
    try:
        request = urllib.request.Request("http://127.0.0.1:8080/health", headers={"Authorization": "Bearer " + LOCAL_LLM_API_KEY})
        with urllib.request.urlopen(request, timeout=5) as response:
            if response.status == 200: break
    except (urllib.error.URLError, TimeoutError):
        pass
    if time.monotonic() > deadline:
        stop_owned_process(server_process)
        raise TimeoutError("10分以内にモデルが起動しませんでした。ログを確認してください。")
    time.sleep(2)
print("モデルAPI起動完了")

In [ ]:
# @title 7. 診断（必要な場合のみ）
# キーを出力に残さず末尾だけ確認。
print((RUNTIME / "llama-server.log").read_text(errors="replace")[-5000:].replace(LOCAL_LLM_API_KEY, "[redacted]"))

In [ ]:
# @title 8. ツール呼び出しの動作確認
def model_request(payload):
    request = urllib.request.Request("http://127.0.0.1:8080/v1/chat/completions",
        data=json.dumps({"model": "qwen-local", **payload}).encode(),
        headers={"Content-Type": "application/json", "Authorization": "Bearer " + LOCAL_LLM_API_KEY})
    with urllib.request.urlopen(request, timeout=300) as response:
        return json.load(response)
smoke = model_request({"messages": [{"role": "user", "content": "Call echo with value ready."}],
    "tools": [{"type": "function", "function": {"name": "echo", "description": "Echo a value",
    "parameters": {"type": "object", "properties": {"value": {"type": "string"}}, "required": ["value"]}}}],
    "tool_choice": "required", "max_tokens": 1024, "temperature": 0.1,
    "chat_template_kwargs": {"enable_thinking": False}})
call = smoke["choices"][0]["message"].get("tool_calls", [])
assert call and call[0]["function"]["name"] == "echo", "ツール呼び出しに失敗。実装は開始せずログと版を確認してください。"
assert json.loads(call[0]["function"]["arguments"])["value"] == "ready"
print("APIとツール呼び出し: OK（実装品質の合格ではありません）")

In [ ]:
# @title 9. Issueと作業ブランチを準備・前回作業を復元
from google.colab import userdata
GH_TOKEN_VALUE = userdata.get("GH_TOKEN")
github_env = os.environ.copy()
github_env.update({"GH_TOKEN": GH_TOKEN_VALUE, "GH_CONFIG_DIR": str(RUNTIME / "gh-config")})
def gh(args):
    return run(["gh", *args], env=github_env, capture=True)
issue = json.loads(gh(["issue", "view", str(ISSUE_NUMBER), "--repo", REPOSITORY,
    "--json", "number,title,body,comments,state,labels"]))
assert issue["state"] == "OPEN", "完了済みIssueです。再実装せず別Issueを指定してください。"
pulls = json.loads(gh(["pr", "list", "--repo", REPOSITORY, "--state", "open",
    "--json", "number,title,body,headRefName,url"]))
related = [p for p in pulls if re.search(rf"(?<![0-9])#{ISSUE_NUMBER}(?![0-9])", p["body"] + " " + p["title"])]
if related:
    print("関連候補PR:", [(p["number"], p["headRefName"], p["url"]) for p in related])
CHECKPOINTS = CACHE / "work" / f"issue-{ISSUE_NUMBER}"
saved_branch = read_json(CHECKPOINTS / "latest.json")["branch"] if RESUME_WORK and (CHECKPOINTS / "latest.json").exists() else ""
branch = choose_work_branch(ISSUE_NUMBER, WORK_BRANCH, saved_branch, related)
print("作業ブランチを選択:", branch)
run(["git", "check-ref-format", "--branch", branch], capture=True)
assert branch not in ("main", "master"), "専用ブランチを指定してください"
REPO = RUNTIME / f"issue-{ISSUE_NUMBER}"
CHECKPOINTS = CACHE / "work" / f"issue-{ISSUE_NUMBER}"
remote = f"https://github.com/{REPOSITORY}.git"
if not REPO.exists():
    if RESUME_WORK and saved_branch == branch and (CHECKPOINTS / "latest.json").exists():
        restored = restore_checkpoint(CHECKPOINTS, REPO, remote)
        assert restored["branch"] == branch, "保存branchと設定が異なります。WORK_BRANCHを合わせてください。"
        print("作業を復元:", restored["id"])
    else:
        run(["gh", "auth", "setup-git"], env=github_env)
        run(["git", "clone", remote, str(REPO)], env=github_env)
        existing = run(["git", "branch", "-r", "--list", f"origin/{branch}"], cwd=REPO, capture=True).strip()
        if existing:
            run(["git", "switch", "--track", f"origin/{branch}"], cwd=REPO)
        else:
            assert not WORK_BRANCH, "指定branchがありません。継続先を確認してください。"
            run(["git", "switch", "-c", branch, "origin/main"], cwd=REPO)
assert run(["git", "branch", "--show-current"], cwd=REPO, capture=True).strip() == branch
assert run(["git", "remote", "get-url", "origin"], cwd=REPO, capture=True).strip() == remote
run(["git", "config", "user.name", "Colab implementation agent"], cwd=REPO)
run(["git", "config", "user.email", "colab-agent@users.noreply.github.com"], cwd=REPO)
run(["npm", "ci"], cwd=REPO)
CONTROL = prepare_control_folder(REPO)
(CONTROL / "workflow.md").write_text(QWEN_WORKFLOW_INSTRUCTIONS, encoding="utf-8")
ISSUE_CONTEXT = CONTROL / "issue-context.json"
def refresh_issue_context():
    current_issue = json.loads(gh(["issue", "view", str(ISSUE_NUMBER), "--repo", REPOSITORY,
        "--json", "number,title,body,comments,state,labels"]))
    assert current_issue["state"] == "OPEN", "対象Issueが完了済みです。再実装しません。"
    current_prs = json.loads(gh(["pr", "list", "--repo", REPOSITORY, "--head", branch, "--state", "open",
        "--json", "number,title,body,headRefName,headRefOid,url,statusCheckRollup"]))
    failures = []
    for pr in current_prs:
        for check in pr.get("statusCheckRollup", []):
            if check.get("conclusion") in ("FAILURE", "TIMED_OUT", "CANCELLED", "ACTION_REQUIRED") or check.get("state") in ("FAILURE", "ERROR"):
                match = re.match(rf"https://github.com/{re.escape(REPOSITORY)}/actions/runs/([0-9]+)", check.get("detailsUrl", ""))
                log = "失敗ログの自動取得ができませんでした。未確認として扱ってください。"
                if match:
                    try: log = gh(["run", "view", match[1], "--repo", REPOSITORY, "--log-failed"])[-60000:]
                    except subprocess.CalledProcessError: pass
                failures.append({"pr": pr["number"], "head": pr["headRefOid"], "check": check, "log": log})
    write_json(ISSUE_CONTEXT, {"issue": current_issue, "related_prs": current_prs or related})
    write_json(CONTROL / "ci-failures.json", {"failures": failures})
    return current_issue
issue = refresh_issue_context()
write_json(Path.home() / ".qwen/settings.json", agent_settings(CONTEXT_SIZE))
checkpoint(REPO, CHECKPOINTS, {"phase": "prepared", "model": MODEL_METADATA["revision"]})
print(issue["title"], "branch:", branch)

In [ ]:
# @title 10. 実装・独立レビュー・修正する（RUN_AGENT=Trueで有効）
import threading
if RUN_AGENT:
    WORKFLOW_STATE = {"ready": False}
    issue = refresh_issue_context()
    agent_env = {k: v for k, v in os.environ.items()
                 if not any(word in k.upper() for word in ("TOKEN", "SECRET", "PASSWORD", "API_KEY"))}
    agent_env["LOCAL_LLM_API_KEY"] = LOCAL_LLM_API_KEY
    stop_checkpoints = threading.Event()
    def periodic_save():
        while not stop_checkpoints.wait(120):
            try: checkpoint(REPO, CHECKPOINTS, {"phase": "in-progress"})
            except Exception as error: print("途中保存に失敗:", type(error).__name__)
    saver = threading.Thread(target=periodic_save, daemon=True)
    saver.start()
    WORKFLOW_DIR = CHECKPOINTS / ("workflow-" + time.strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:8])
    write_json(CONTROL / "active-workflow.json", {"directory": str(WORKFLOW_DIR)})
    WORKFLOW_STATE = {"ready": False}
    try:
        WORKFLOW_STATE = run_issue_workflow(REPO, WORKFLOW_DIR, ISSUE_NUMBER, ISSUE_CONTEXT,
            agent_env, CONTEXT_SIZE, MAX_TURNS, MAX_MINUTES, MAX_REPAIRS, USER_FEEDBACK)
        print("状態:", WORKFLOW_STATE["phase"])
        if WORKFLOW_STATE["ready"]:
            print("Qwen独立レビューと必須チェック完了。ユーザーの動作確認へ進めます。")
        else:
            print("指摘/未完了が残っています。合格ではありません。差分を保存しました。")
        last_review = WORKFLOW_STATE["rounds"][-1]["review"].get("report") if WORKFLOW_STATE["rounds"] else None
        if last_review:
            print("画面で確認する操作:")
            for index, item in enumerate(last_review.get("user_checks", []), 1):
                print(f"{index}. {item['action']} → {item['expected']}")
            for item in last_review.get("unchecked", []): print("未検査:", item)
    finally:
        stop_checkpoints.set(); saver.join(timeout=30)
        checkpoint(REPO, CHECKPOINTS, {"phase": "workflow-stopped", "model": MODEL_METADATA["revision"]})
    print("保存先:", CHECKPOINTS)
else:
    print("RUN_AGENT=Trueにして実行します。修正依頼はUSER_FEEDBACKへ普通の言葉で記入します。")

In [ ]:
# @title 11. 保存・差分確認（いつでも実行可能）
print(run(["git", "status", "--short"], cwd=REPO, capture=True))
checkpoint(REPO, CHECKPOINTS, {"phase": "manual-save"})
handoff = REPO / ".COLAB_HANDOFF.md"
if handoff.exists(): print(handoff.read_text(encoding="utf-8"))
print("作業保存済み。ログ/途中保存はDriveに残ります。")

In [ ]:
# @title 12. 検証と画面プレビュー（エージェント終了後）
require_reviewed_workspace(REPO, WORKFLOW_STATE, ISSUE_NUMBER)
run(["npm", "run", "check"], cwd=REPO)
run(["npm", "run", "references:check"], cwd=REPO)
PREVIEW_BUILD = read_json(REPO / "dist/build-info.json")
require_reviewed_workspace(REPO, WORKFLOW_STATE, ISSUE_NUMBER)
PREVIEW_SHA = PREVIEW_BUILD["commit"]
PREVIEW_CLEAN = not PREVIEW_BUILD["dirty"]
CHECKED_DIFF = run(["git", "diff", "--binary", "HEAD"], cwd=REPO, capture=True)
if "preview_process" in globals() and preview_process.poll() is None and globals().get("preview_repo") != REPO:
    stop_owned_process(preview_process)
if "preview_process" not in globals() or preview_process.poll() is not None:
    preview_log = open(RUNTIME / "preview.log", "w")
    preview_process = subprocess.Popen(["npm", "run", "preview", "--", "--port", "5182", "--strictPort"],
        cwd=REPO, stdout=preview_log, stderr=subprocess.STDOUT, start_new_session=True)
    preview_repo = REPO
    time.sleep(3)
    assert preview_process.poll() is None, "プレビュー起動失敗。preview.logを確認してください。"
from google.colab import output
output.serve_kernel_port_as_iframe(5182, height=650)
print("このColab内の画面です。ブラウザ操作・900×420・タッチなどは別途確認してください。")

## PRを作る場合
次のセルはチェックを再実行してから、専用ブランチをpushし、PRを作成/更新します。
`PUBLISH_PR=True`を選んだ場合だけ外部へ反映します。mainへ直接pushせず、マージもしません。
既存PRがある場合は新規PRを重複作成しません。
CI失敗時はセル10を再実行すると、最新コメントとCI失敗ログをQwenへ渡して修正・再レビューします。
技術確認は親セルとQwenレビューが担当します。ユーザーはセル12の操作案内に沿ってふるまいを確認します。

In [ ]:
# @title 13. PRを作成/更新する（任意）
PUBLISH_PR = False # @param {type:"boolean"}
PR_TITLE = "" # @param {type:"string"}
PR_DESCRIPTION = "" # @param {type:"string"}
if PUBLISH_PR:
    require_reviewed_workspace(REPO, WORKFLOW_STATE, ISSUE_NUMBER)
    assert run(["git", "branch", "--show-current"], cwd=REPO, capture=True).strip() == branch
    assert branch not in ("main", "master")
    assert json.loads(gh(["issue", "view", str(ISSUE_NUMBER), "--repo", REPOSITORY, "--json", "state"]))["state"] == "OPEN", "対象Issueが完了済みです。重複実装を提出しないでください。"
    run(["npm", "run", "check"], cwd=REPO)
    run(["npm", "run", "references:check"], cwd=REPO)
    require_reviewed_workspace(REPO, WORKFLOW_STATE, ISSUE_NUMBER)
    run(["git", "add", "--all"], cwd=REPO)
    if run(["git", "diff", "--cached", "--name-only"], cwd=REPO, capture=True).strip():
        run(["git", "commit", "-m", PR_TITLE or issue["title"]], cwd=REPO)
    run(["npm", "run", "build"], cwd=REPO)
    assert not read_json(REPO / "dist/build-info.json")["dirty"], "検査版はclean commitから作成してください"
    require_reviewed_workspace(REPO, WORKFLOW_STATE, ISSUE_NUMBER)
    run(["gh", "auth", "setup-git"], env=github_env)
    run(["git", "push", "--set-upstream", "origin", branch], cwd=REPO, env=github_env)
    PUBLISHED_SHA = run(["git", "rev-parse", "HEAD"], cwd=REPO, capture=True).strip()
    existing = json.loads(gh(["pr", "list", "--repo", REPOSITORY, "--head", branch,
                              "--state", "open", "--json", "number,url"]))
    body = RUNTIME / "pr-body.md"
    handoff_file = REPO / ".COLAB_HANDOFF.md"
    description = PR_DESCRIPTION.strip() or (handoff_file.read_text(encoding="utf-8") if handoff_file.exists()
        else "実装エージェントの引き継ぎが未記載です。完了扱いせず、差分と未完了事項の確認が必要です。")
    review = WORKFLOW_STATE["rounds"][-1]["review"]["report"]
    description += "\n\nQwen独立レビュー: pass（別会話・読み取り専用）。\n"
    description += "\n".join(f"- {item['action']} → {item['expected']}" for item in review["user_checks"])
    description += "\n未検査: " + " / ".join(review["unchecked"])
    body.write_text(description + f"\n\nRelated: #{ISSUE_NUMBER}\n\n検証: npm run check / npm run references:check成功。ブラウザ実操作は別途確認。\nモデル: {MODEL_REPO} @ {MODEL_METADATA['revision']} / llama.cpp @ {LLAMA_COMMIT}\nユーザー合格後にマージ。", encoding="utf-8")
    if existing:
        print(gh(["pr", "edit", str(existing[0]["number"]), "--repo", REPOSITORY, "--body-file", str(body)]))
    else:
        print(gh(["pr", "create", "--draft", "--repo", REPOSITORY, "--base", "main", "--head", branch,
                  "--title", PR_TITLE or issue["title"], "--body-file", str(body)]))
    checkpoint(REPO, CHECKPOINTS, {"phase": "pr-published"})
    print("PR提出SHA:", PUBLISHED_SHA, "。セル12を再実行して、このclean版を画面で確認してください。")
else:
    print("GitHubへの反映は行っていません。作業はDriveへ保存できます。")

## 動作確認で合格した場合のみ
PR提出後にセル12を再実行し、cleanな対象版を画面で確認してください。
修正が必要ならセル1のUSER_FEEDBACKへ普通の言葉で記入し、セル10からやり直します。
合格した場合だけ次のUSER_ACCEPTEDを有効にします。PRのheadと検査版SHAが同じでCI成功ならマージします。

In [ ]:
# @title 14. ユーザー検査に合格した版をマージする（任意）
USER_ACCEPTED = False # @param {type:"boolean"}
if USER_ACCEPTED:
    require_reviewed_workspace(REPO, WORKFLOW_STATE, ISSUE_NUMBER)
    expected_sha = run(["git", "rev-parse", "HEAD"], cwd=REPO, capture=True).strip()
    candidates = json.loads(gh(["pr", "list", "--repo", REPOSITORY, "--head", branch, "--state", "open", "--json", "number"]))
    assert len(candidates) == 1, "対象PRを一意に確認できません。"
    pr_number = candidates[0]["number"]
    pr = json.loads(gh(["pr", "view", str(pr_number), "--repo", REPOSITORY,
        "--json", "headRefOid,isDraft,statusCheckRollup"]))
    expected_sha = require_approved_revision(REPO, WORKFLOW_STATE, ISSUE_NUMBER, PREVIEW_BUILD, PUBLISHED_SHA, pr)
    if pr["isDraft"]: gh(["pr", "ready", str(pr_number), "--repo", REPOSITORY])
    print(gh(["pr", "merge", str(pr_number), "--repo", REPOSITORY, "--squash", "--match-head-commit", expected_sha]))
    merged = json.loads(gh(["pr", "view", str(pr_number), "--repo", REPOSITORY, "--json", "state"]))
    assert merged["state"] == "MERGED", "マージ未完了です。Issueは閉じません。"
    gh(["issue", "close", str(ISSUE_NUMBER), "--repo", REPOSITORY, "--reason", "completed",
        "--comment", f"ユーザーがNotebookで検査合格を明示。検査対象 {expected_sha} / PR #{pr_number} をマージ済み。"])
    checkpoint(REPO, CHECKPOINTS, {"phase": "merged", "approved_sha": expected_sha})
    print("合格版をマージしIssueを完了しました。")
else:
    print("まだマージしていません。ゲームの動作確認で合格した場合だけUSER_ACCEPTED=Trueにします。")

In [ ]:
# @title 15. 終了（保存してGPUを解放）
for name in ("agent_process", "preview_process", "server_process"):
    process = globals().get(name)
    if process is not None and process.poll() is None:
        stop_owned_process(process)
if "REPO" in globals() and REPO.exists(): checkpoint(REPO, CHECKPOINTS, {"phase": "shutdown"})
print("保存・プロセス停止済み。Colabメニューでランタイムを切断してください。")

## 再開・トラブル
- 次回も同じDriveフォルダ・Issue番号・WORK_BRANCHで、セル1から順に実行します。
  不具合はUSER_FEEDBACKへ操作と困った結果を記入し、セル10を再実行します。
  Driveのモデルは再ダウンロードせず、同じ版を使用します。作業差分はセル9で復元します。
- 16GB GPUでは標準27B構成を開始しません。GPUを変更してください。
- モデル破損が疑われる場合はREPAIR_MODEL=Trueで再実行します。
- llama.cppを更新する場合はLLAMA_REFを新しいtag/refへ変更します。
  モデル更新はMODEL_REVISIONに新しいSHAを指定します。元のキャッシュは保持します。
- 途中保存は世代ごとに残ります。Drive容量が不足したら、不要になった古いwork世代を手動で整理してください。
- 許可されていないシェルコマンドは実行が拒否されます。ログと引き継ぎを確認し、必要なコマンドだけ
  agent_settingsのallowへ追加してください。権限の一括スキップは使用していません。
- 切断後の再開はコード差分と引き継ぎから行います。Qwenの会話履歴そのものは復元しません。
- ログ・変更ファイルは私有Driveへ保存されます。Secretsをソース/ログへ書く指示を出さないでください。

参照: [モデル配布](https://huggingface.co/unsloth/Qwen3.8-27B-GGUF) /
[Qwen Code](https://github.com/QwenLM/qwen-code) /
[Qwenサブエージェント](https://qwenlm.github.io/qwen-code-docs/en/users/features/sub-agents/) /
[llama.cpp](https://github.com/ggml-org/llama.cpp/tree/master/tools/server) /
[Colabの制限](https://research.google.com/colaboratory/faq.html)